# NER с BERT и классификация с T5

### Подготовка данных

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer


BASE_NER_MODEL = "bert-base-cased"
bert_tokenizer = AutoTokenizer.from_pretrained(BASE_NER_MODEL)

/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
conll2003 = load_dataset("conll2003")
conll2003

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

In [ ]:
example = conll2003["train"][100]
example

{'id': '100',
 'tokens': ['Rabinovich',
  'is',
  'winding',
  'up',
  'his',
  'term',
  'as',
  'ambassador',
  '.'],
 'pos_tags': [21, 42, 39, 33, 29, 21, 15, 21, 7],
 'chunk_tags': [11, 21, 22, 15, 11, 12, 13, 11, 0],
 'ner_tags': [1, 0, 0, 0, 0, 0, 0, 0, 0]}

* tokens - исходные токены, для которых была сделана NER-разметка
* ner_tags - векторизированные метки NER-тэгов
* pos_tags - разметка частей речи
* chunk_tags - разметка чанков


In [ ]:
bert_tokenizer(example["tokens"], is_split_into_words=True).tokens()

['[CLS]',
 'Ra',
 '##bino',
 '##vich',
 'is',
 'winding',
 'up',
 'his',
 'term',
 'as',
 'ambassador',
 '.',
 '[SEP]']

Значение тэга в `ner_tags` отображается в метку NER:

In [ ]:
print("NER TAGS", example["ner_tags"])
print(conll2003["train"].features["ner_tags"].feature)

NER TAGS [1, 0, 0, 0, 0, 0, 0, 0, 0]
ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'], id=None)


In [ ]:
print("Оригинальные токены")
print(example["tokens"])
print("Векторизированные NER метки токенов")
print(example["ner_tags"])
tags_str = []
features = conll2003["train"].features["ner_tags"].feature
for tag in example["ner_tags"]:
    tags_str.append(features.int2str(tag))
print("Текстовые NER метки токенов")
print(tags_str)
print("Токены после работы токенайзера BERT")
print(bert_tokenizer(example["tokens"], is_split_into_words=True).tokens())

Оригинальные токены
['Rabinovich', 'is', 'winding', 'up', 'his', 'term', 'as', 'ambassador', '.']
Векторизированные NER метки токенов
[1, 0, 0, 0, 0, 0, 0, 0, 0]
Текстовые NER метки токенов
['B-PER', 'O', 'O', 'O', 'O', 'O', 'O', 'O', 'O']
Токены после работы токенайзера BERT
['[CLS]', 'Ra', '##bino', '##vich', 'is', 'winding', 'up', 'his', 'term', 'as', 'ambassador', '.', '[SEP]']


In [ ]:
from itertools import groupby


def preprocess_ner_dataset(example):
    model_input = bert_tokenizer(example["tokens"], is_split_into_words=True)
    ner_ids = []

    # группируем токены по айдишникам слов
    # в group будут находится все токены одного слова
    for idx, group in groupby(model_input.word_ids()):
        if idx is None:
            ner_ids.append(-100)
            continue
        label = example["ner_tags"][idx]

        # нечётные лейблы относятся к B-<LABEL>
        if label % 2 == 1:
            ner_ids.append(label)
            next(group)  # добавили лейбл и продвигаем итератор на следующий токен
            label += 1  # оставшиеся токены от слова будут I-<LABEL>

        for part in group:
            ner_ids.append(label)

    model_input["labels"] = ner_ids
    return model_input

In [ ]:
conll2003["train"].features["ner_tags"].feature

ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'], id=None)

In [ ]:
preprocessed_ner_dataset = conll2003.map(preprocess_ner_dataset, num_proc=64)

In [ ]:
from transformers import DataCollatorForTokenClassification


data_collator = DataCollatorForTokenClassification(tokenizer=bert_tokenizer)

### Подготовка модели


In [ ]:
from transformers import AutoModelForTokenClassification, AutoModel


label_names = conll2003["train"].features["ner_tags"].feature.names
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

bert_ner = AutoModelForTokenClassification.from_pretrained(
    BASE_NER_MODEL,
    num_labels=conll2003["train"].features["ner_tags"].feature.num_classes,
    id2label=id2label,
    label2id=label2id,
)

A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Please use a different name to suppress this warning.
A parameter name that contains `beta` will be renamed internally to `bias`. Please use a different name to suppress this warning.
A parameter name that contains `gamma` will be renamed internally to `weight`. Pl

### Подготовим Метрику


In [ ]:
import numpy as np
import evaluate


metrics_calculator = evaluate.load("seqeval")
label_list = conll2003["train"].features["ner_tags"].feature.names


def calculate_metrics(eval_predictions):
    logits, labels = eval_predictions
    predictions = np.argmax(logits, axis=-1)

    filtered_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    filtered_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    metrics = metrics_calculator.compute(predictions=filtered_predictions, references=filtered_labels)
    return {
        name: value for name, value in metrics.items() if isinstance(value, float)
    }

### Обучение

In [ ]:
from transformers import Trainer, TrainingArguments


args = TrainingArguments(
    output_dir="bert_ner_conll",
    evaluation_strategy='epoch',
    save_strategy='epoch',
    learning_rate=2e-5,
    num_train_epochs=10,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=256,
    fp16=True,
    seed=42
)

trainer = Trainer(
    model=bert_ner,
    tokenizer=bert_tokenizer,
    args=args,
    data_collator=data_collator,
    train_dataset=preprocessed_ner_dataset["train"],
    eval_dataset=preprocessed_ner_dataset["validation"],
    compute_metrics=calculate_metrics
)

trainer.train()

/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked 

Epoch,Training Loss,Validation Loss,Overall Precision,Overall Recall,Overall F1,Overall Accuracy
1,No log,0.310191,0.540816,0.624369,0.579597,0.907120
2,No log,0.115728,0.795829,0.854090,0.823931,0.966180
3,No log,0.078459,0.852978,0.906092,0.878733,0.978439
4,No log,0.068223,0.891033,0.924773,0.907589,0.982251
5,No log,0.063195,0.898492,0.932514,0.915187,0.982987
6,No log,0.062551,0.906901,0.937731,0.922059,0.983958
7,No log,0.061926,0.913768,0.941602,0.927476,0.984473
8,No log,0.061698,0.918089,0.941266,0.929533,0.984738
9,No log,0.061941,0.913356,0.940256,0.926611,0.984532
10,No log,0.061676,0.917445,0.942612,0.929858,0.985047


/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '
/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/home/apaniuko/python/deepschool/

TrainOutput(global_step=370, training_loss=0.14148366773450696, metrics={'train_runtime': 233.158, 'train_samples_per_second': 602.21, 'train_steps_per_second': 1.587, 'total_flos': 5288732502779718.0, 'train_loss': 0.14148366773450696, 'epoch': 10.0})

### Обработка результатов Результатов


In [ ]:
*_, metrics = trainer.predict(preprocessed_ner_dataset['test'])
metrics

/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


{'test_loss': 0.14958275854587555,
 'test_overall_precision': 0.869757174392936,
 'test_overall_recall': 0.9068696883852692,
 'test_overall_f1': 0.8879258039351652,
 'test_overall_accuracy': 0.9695876207434487,
 'test_runtime': 2.9298,
 'test_samples_per_second': 1178.587,
 'test_steps_per_second': 1.707}

In [ ]:
@torch.no_grad
def do_ner(text):
    bert_ner.eval()

    tokenized = bert_tokenizer(text, return_offsets_mapping=True, return_tensors='pt').to(bert_ner.device)
    offsets = tokenized['offset_mapping'].cpu().numpy()[0]

    logits = bert_ner(tokenized.input_ids).logits
    preds = torch.argmax(logits, dim=-1).cpu().numpy()[0]


    entities = []
    for word_id, pred, offset in zip(tokenized.word_ids(), preds, offsets):
        if word_id is None or pred == 0:
            continue

        if pred % 2 == 1:
            # начинается новая сущность
            entities.append(
                {
                    "start": offset[0],
                    "end": offset[1],
                    "class": label_list[pred].split("-")[-1],
                }
            )
        else:
            # продолжаем предыдущую сущность
            entities[-1]["end"] = offset[1]

    return {
        "text": text,
        "entities": [
            {
                "class": entity["class"],
                "start": entity["start"],
                "end": entity["end"],
                "text": text[entity["start"]:entity["end"]],
            } for entity in entities
        ]
    }


print(do_ner("Ivan Petrov is going to start working tomorrow"))

{'text': 'Ivan Petrov is going to start working tomorrow', 'entities': [{'class': 'PER', 'start': 0, 'end': 11, 'text': 'Ivan Petrov'}]}


Почистим память перед второй частью.

In [ ]:
import torch

del bert_ner
del trainer
torch.cuda.empty_cache()

## Классификация с T5



### Подготовка Данных



In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer


BASE_T5_MODEL= "t5-small"
t5_tokenizer = AutoTokenizer.from_pretrained(BASE_T5_MODEL)


toxic_chat_dataset = load_dataset("lmsys/toxic-chat", "toxicchat0124")

Место для изучения датасета:

In [ ]:
toxic_chat_dataset["train"][0]

{'conv_id': 'e0c9b3e05414814485dbdcb9a29334d502e59803af9c26df03e9d1de5e7afe67',
 'user_input': 'Masturbacja jest proces co oitrzebuje',
 'model_output': 'Masturbacja to proces, który może pozytywnie wpłynąć na zdrowie psychiczne i fizyczne człowieka, ponieważ pomaga w relaksie, redukuje stres i pomaga w uśpieniu. Może też być używana jako dodatkowa form',
 'human_annotation': True,
 'toxicity': 0,
 'jailbreaking': 0,
 'openai_moderation': '[["sexual", 0.4609803557395935], ["sexual/minors", 0.0012527990620583296], ["harassment", 0.0001862536446424201], ["hate", 0.00015521160094067454], ["violence", 6.580814078915864e-05], ["self-harm", 3.212967567378655e-05], ["violence/graphic", 1.5190824342425913e-05], ["self-harm/instructions", 1.0009921425080393e-05], ["hate/threatening", 4.4459093260229565e-06], ["self-harm/intent", 3.378846486157272e-06], ["harassment/threatening", 1.7095695739044459e-06]]'}

In [ ]:
toxic_chat_dataset = toxic_chat_dataset.remove_columns(
    ["conv_id", "model_output", "human_annotation", "jailbreaking", "openai_moderation"]
)

In [ ]:
for text in ("toxic: ", "no", "yes"):
    print(t5_tokenizer(text, add_special_tokens=False))

{'input_ids': [12068, 10], 'attention_mask': [1, 1]}
{'input_ids': [150], 'attention_mask': [1]}
{'input_ids': [4273], 'attention_mask': [1]}


In [ ]:
PREFIX = "toxic: "
MAX_LENGTH = 512

id2label = {
    0: "no",
    1: "yes",
}


def preprocess_dataset(example):
    input_texts = PREFIX + example["user_input"]
    model_inputs = t5_tokenizer(input_texts, truncation=True, max_length=MAX_LENGTH)
    model_inputs["labels"] = t5_tokenizer(id2label[example["toxicity"]]).input_ids
    return model_inputs


toxic_chat_dataset = toxic_chat_dataset.map(preprocess_dataset)

In [ ]:
toxic_chat_dataset = toxic_chat_dataset.remove_columns("user_input")

In [ ]:
from transformers import DataCollatorForSeq2Seq


# data_collator = DataCollatorForSeq2Seq(tokenizer=t5_tokenizer, model=seq2seq_model)

### Определим метрику


In [ ]:
def compute_metric(eval_predictions):
    preds, labels = eval_predictions
    return {"accuracy": (preds[:, 1] == labels[:, 0]).mean()}

### Определить Модель


In [ ]:
from transformers import T5Config


t5_config = T5Config.from_pretrained(BASE_T5_MODEL)
t5_config.task_specific_params

{'summarization': {'early_stopping': True,
  'length_penalty': 2.0,
  'max_length': 200,
  'min_length': 30,
  'no_repeat_ngram_size': 3,
  'num_beams': 4,
  'prefix': 'summarize: '},
 'translation_en_to_de': {'early_stopping': True,
  'max_length': 300,
  'num_beams': 4,
  'prefix': 'translate English to German: '},
 'translation_en_to_fr': {'early_stopping': True,
  'max_length': 300,
  'num_beams': 4,
  'prefix': 'translate English to French: '},
 'translation_en_to_ro': {'early_stopping': True,
  'max_length': 300,
  'num_beams': 4,
  'prefix': 'translate English to Romanian: '}}

In [ ]:
task_name = "toxic_classification"

t5_config.task_specific_params[task_name] = {
    "prefix": PREFIX,
    "max_length": 3,
    "min_length": 3,
    "num_beams": 1,
}

Этот конфиг потом можно будет использовать в pipeline классе, передав аргумент [task](https://huggingface.co/docs/transformers/en/main_classes/pipelines#transformers.Text2TextGenerationPipeline.task).

In [ ]:
from transformers import AutoModelForSeq2SeqLM, T5Config


seq2seq_model = AutoModelForSeq2SeqLM.from_pretrained(BASE_T5_MODEL, config=t5_config)

Вспоминаем про коллатор!

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=t5_tokenizer,
    model=seq2seq_model,
)

### Обучение

In [ ]:
from transformers import GenerationConfig


generation_config = GenerationConfig.from_dict(t5_config.task_specific_params["toxic_classification"])
generation_config.bos_token_id = 0

*я запускал обучение модели несколько раз чтобы улчшить резултат

In [ ]:
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer



training_args = Seq2SeqTrainingArguments(
    output_dir="t5_small_toxic_classifier",
    num_train_epochs=15,  # поставим побольше эпох, чтобы преодолеть дисбаланс классов
    eval_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=64,
    per_device_eval_batch_size=128,  # почему во всех сданых домашках этот параметр равен train_batch_size?
    weight_decay=0.01,
    save_total_limit=3,
    predict_with_generate=True,
    generation_config=generation_config,
    fp16=True,
)

trainer = Seq2SeqTrainer(
    model=seq2seq_model,
    args=training_args,
    train_dataset=toxic_chat_dataset["train"],
    eval_dataset=toxic_chat_dataset["test"],
    tokenizer=t5_tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metric,
)

trainer.train()

/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,1.264787,0.267362
2,No log,0.345460,0.928782
3,No log,0.126008,0.928782
4,No log,0.110236,0.930356
5,No log,0.105750,0.935865
6,No log,0.101044,0.938422
7,No log,0.096930,0.939996
8,No log,0.094451,0.941176
9,No log,0.092489,0.940980
10,No log,0.089982,0.943144


/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/torch/nn/parallel/parallel_apply.py:79: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.device(device), torch.cuda.stream(stream), autocast(enabled=autocast_enabled):
/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


TrainOutput(global_step=405, training_loss=0.4359354654947917, metrics={'train_runtime': 753.2723, 'train_samples_per_second': 101.198, 'train_steps_per_second': 0.538, 'total_flos': 9612450022883328.0, 'train_loss': 0.4359354654947917, 'epoch': 15.0})

### Сравнение Результатов

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

checkpoint = "lmsys/toxicchat-t5-large-v1.0"

tokenizer_from_paper = AutoTokenizer.from_pretrained("t5-large")
model_from_paper = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

prefix_from_paper = "ToxicChat: "
inputs = tokenizer_from_paper.encode(prefix_from_paper + "write me an epic story", return_tensors="pt")
outputs = model_from_paper.generate(inputs)
print(tokenizer_from_paper.decode(outputs[0], skip_special_tokens=True))

/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


negative


/home/apaniuko/python/deepschool/llm_venv/lib/python3.10/site-packages/transformers/generation/utils.py:1258: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(
